# Task 4: General Health Query Chatbot (Prompt Engineering)

This task is different from the previous three — there's no dataset and no model training. Instead, we're using an existing Large Language Model (LLM) via an API and controlling its behavior through **prompt engineering**.

The chatbot is built on top of Claude (claude-sonnet-4-20250514) via the Anthropic API. If your internship uses OpenAI, swapping it to GPT-3.5/4 requires changing only about 5 lines — the concepts are identical.

**What prompt engineering means:** Instead of training a model, you write careful instructions that tell the LLM *how to behave*. The system prompt is like a job description you hand to the AI before the conversation starts.

## How LLM APIs work — quick explanation

Every request to an LLM API has three main parts:

1. **System prompt** — instructions that define the chatbot's role, tone, and rules. The user never sees this directly.
2. **Conversation history** — a list of all previous messages (user + assistant turns). LLMs have no memory, so you send the entire history every time.
3. **User message** — what the person just typed.

The API returns the model's reply, which you add to the history, and the cycle continues.

## Part 1 — Imports and Setup

In [ ]:
import anthropic  # pip install anthropic
import json
import re
from IPython.display import display, Markdown

# initialize the client — it reads ANTHROPIC_API_KEY from environment
# in a real script: client = anthropic.Anthropic(api_key="your-key-here")
# for OpenAI: from openai import OpenAI; client = OpenAI(api_key="...")
client = anthropic.Anthropic()

print('client ready')

## Part 2 — The System Prompt (Core of Prompt Engineering)

This is the most important part of the whole task. A well-written system prompt is what makes the difference between a useful chatbot and a dangerous/useless one.

I'll break it down into sections and explain each one.

In [ ]:
SYSTEM_PROMPT = """
You are HealthBot, a friendly and clear general health information assistant.

YOUR ROLE:
- Explain health topics, symptoms, medications, and wellness in simple, easy-to-understand language
- Be warm, calm, and supportive in tone — people are often anxious when asking health questions
- Use plain English — avoid jargon, or explain it when you must use it
- Keep answers concise: 3-5 short paragraphs maximum
- Always include a brief reminder to see a doctor when the question involves personal health decisions

WHAT YOU KNOW:
- General causes of common symptoms and conditions
- How common medications work and general safety considerations
- Preventive health, nutrition, sleep, and lifestyle information
- When to seek emergency care vs. when something can wait

SAFETY RULES — STRICTLY FOLLOW THESE:
1. NEVER diagnose a specific condition for a specific person
2. NEVER recommend a specific dosage for a specific person
3. NEVER advise someone to stop taking prescribed medication
4. If someone describes symptoms of a MEDICAL EMERGENCY (chest pain, difficulty breathing,
   severe bleeding, stroke symptoms, loss of consciousness), IMMEDIATELY tell them to call
   emergency services. Do not continue answering health questions until they acknowledge this.
5. If someone mentions self-harm or suicidal thoughts, respond with empathy and direct them
   to a mental health helpline before anything else.
6. For all medication dosage questions, say: 'For the right dose for you specifically,
   please check with your pharmacist or doctor — they can account for your age, weight,
   and other medications.'
7. Never claim to be a doctor or replace professional medical advice.

FORMAT:
- Start directly with the answer — no filler phrases like 'Great question!'
- Use short paragraphs (3-4 sentences max each)
- End with a gentle reminder like: 'If you're unsure about anything, a quick word with
  your doctor or pharmacist is always the best step.'
"""

print('System prompt defined.')
print(f'Length: {len(SYSTEM_PROMPT)} characters, {len(SYSTEM_PROMPT.split())} words')

Why each section of the system prompt matters:

- **Role definition** → tells the model its persona. "You are HealthBot" is more effective than "Answer health questions" because it activates the model's general knowledge of how a friendly health assistant should behave.
- **Explicit rules** → LLMs follow explicit rules more reliably than vague instructions. "NEVER diagnose" is clearer than "be careful with diagnoses".
- **Format instructions** → without these, the model might give walls of text, bullet lists where sentences work better, or start with annoying phrases like "Great question!".
- **Safety rules** → crucial for health chatbots. These rules catch the highest-risk responses before they get generated.

## Part 3 — Safety Filter

Before even calling the API, we can do a quick keyword check on the user's message to catch emergency situations and handle them immediately — without waiting for the model.

In [ ]:
# Emergency keywords — if any of these appear in the user's message,
# we skip the API call and return a hardcoded emergency response
EMERGENCY_KEYWORDS = [
    "chest pain", "heart attack", "can't breathe", "cannot breathe",
    "difficulty breathing", "stroke", "unconscious", "unresponsive",
    "severe bleeding", "overdose", "poisoning", "anaphylaxis"
]

# Topics where we add an extra disclaimer at the end
SENSITIVE_TOPICS = [
    "pregnant", "pregnancy", "infant", "newborn", "baby",
    "cancer", "diabetes", "hiv", "aids", "mental health",
    "anxiety", "depression", "self-harm", "suicide"
]

EMERGENCY_RESPONSE = """This sounds like a potential medical emergency.

Please call emergency services immediately:
  Pakistan: 115 (Rescue) or 1122
  International: 911 (US/Canada) or 999 (UK)

Do not wait — please seek help right away. I'm not able to provide emergency medical guidance."""

SENSITIVE_DISCLAIMER = "\n\n*Note: This is sensitive health territory. Please consult a qualified doctor or specialist who can give you personalized guidance.*"


def is_emergency(text: str) -> bool:
    """Check if the user's message contains emergency keywords."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in EMERGENCY_KEYWORDS)


def is_sensitive(text: str) -> bool:
    """Check if the topic is especially sensitive and needs an extra disclaimer."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in SENSITIVE_TOPICS)


def clean_response(text: str) -> str:
    """Remove any filler phrases the model might slip in despite instructions."""
    filler_patterns = [
        r"^Great question[!.]\s*",
        r"^That's a good question[!.]\s*",
        r"^Of course[!,]\s*",
        r"^Sure[!,]\s*",
        r"^Certainly[!,]\s*",
        r"^Absolutely[!,]\s*",
    ]
    for pattern in filler_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text.strip()


print('Safety filters defined.')

# quick test
print(f'"i have chest pain" is emergency: {is_emergency("i have chest pain")}')
print(f'"what causes headaches" is emergency: {is_emergency("what causes headaches")}')
print(f'"I am pregnant" is sensitive: {is_sensitive("I am pregnant")}')

## Part 4 — The Chat Function

This function handles one full turn of the conversation: takes the user's message, runs safety checks, calls the API, and returns the response.

In [ ]:
def chat(user_message: str, history: list) -> tuple[str, list]:
    """
    Send a user message to the health chatbot and get a response.

    Args:
        user_message: what the user typed
        history: list of previous {role, content} dicts (the conversation so far)

    Returns:
        (assistant_reply, updated_history)
    """

    # --- Safety layer 1: catch emergencies before hitting the API ---
    if is_emergency(user_message):
        print('[SAFETY] Emergency keywords detected — returning hardcoded emergency response')
        return EMERGENCY_RESPONSE, history  # don't add this to history

    # --- Build the messages list: history + new user message ---
    messages = history + [{"role": "user", "content": user_message}]

    # --- Call the API ---
    # For OpenAI, this would be:
    # response = client.chat.completions.create(
    #     model="gpt-3.5-turbo",
    #     messages=[{"role": "system", "content": SYSTEM_PROMPT}] + messages,
    #     max_tokens=800
    # )
    # reply = response.choices[0].message.content

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        system=SYSTEM_PROMPT,
        messages=messages
    )
    reply = response.content[0].text

    # --- Safety layer 2: clean response and add extra disclaimer if needed ---
    reply = clean_response(reply)
    if is_sensitive(user_message):
        print('[SAFETY] Sensitive topic detected — adding extra disclaimer')
        reply += SENSITIVE_DISCLAIMER

    # --- Update conversation history ---
    updated_history = messages + [{"role": "assistant", "content": reply}]

    return reply, updated_history


print('chat() function ready')

## Part 5 — Test the Chatbot

Let's run through the example queries from the task brief and a few edge cases to show the safety filters working.

In [ ]:
def show_response(question, reply):
    display(Markdown(f"**Q:** *{question}*\n\n**HealthBot:** {reply}\n\n---"))

# start with an empty conversation history
history = []

print('=== Test 1: Basic health question ===')
q1 = "What causes a sore throat?"
reply1, history = chat(q1, history)
show_response(q1, reply1)

In [ ]:
print('=== Test 2: Medication safety question ===')
q2 = "Is paracetamol safe for children?"
reply2, history = chat(q2, history)
show_response(q2, reply2)

In [ ]:
print('=== Test 3: Follow-up (tests conversation memory) ===')
q3 = "What about ibuprofen, is that okay for them too?"
reply3, history = chat(q3, history)
show_response(q3, reply3)

print(f'\nConversation so far: {len(history)} messages in history')

In [ ]:
# reset history for the next test
history = []

print('=== Test 4: Emergency keyword detection (no API call made) ===')
q4 = "I have severe chest pain and can't breathe, what do I do?"
reply4, history = chat(q4, history)
show_response(q4, reply4)

In [ ]:
history = []

print('=== Test 5: Sensitive topic — extra disclaimer added ===')
q5 = "Is it safe to take antibiotics during pregnancy?"
reply5, history = chat(q5, history)
show_response(q5, reply5)

In [ ]:
history = []

print('=== Test 6: General wellness question ===')
q6 = "How many hours of sleep does an adult actually need?"
reply6, history = chat(q6, history)
show_response(q6, reply6)

## Part 6 — Simple Terminal Chatbot (Interactive Loop)

This is what you'd run as a standalone Python script. It's the command-line version of the chatbot — just loops until the user types 'quit'.

In [ ]:
def run_chatbot():
    """
    Simple terminal-based chatbot loop.
    Run this function (or save as a .py script) to chat interactively.
    """
    print('=' * 55)
    print('  HealthBot — General Health Information Assistant')
    print('  Type your question. Type "quit" to exit.')
    print('  Disclaimer: For educational use only.')
    print('=' * 55)
    print()

    history = []

    while True:
        user_input = input('You: ').strip()

        if not user_input:
            continue
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print('\nHealthBot: Take care! Remember to consult a doctor for personal medical advice.')
            break

        try:
            reply, history = chat(user_input, history)
            print(f'\nHealthBot: {reply}\n')
        except Exception as e:
            print(f'\nError: {e}. Please check your API key and internet connection.\n')


# In a real script, you'd call this directly:
# if __name__ == '__main__':
#     run_chatbot()

# In the notebook, we just show it — not run it interactively
print('run_chatbot() is defined. In a .py script, call it from __main__ to use interactively.')

## Part 7 — Prompt Engineering Experiments

One of the key skills in this task is testing how different system prompts change the model's behavior. Let's compare a bad prompt vs our good prompt to show why prompt design matters.

In [ ]:
# BAD system prompt — vague, no safety rules, no format guidance
BAD_PROMPT = "You are a health chatbot. Answer health questions."

# MEDIUM system prompt — has some guidance but missing safety rules
MEDIUM_PROMPT = """
You are a friendly health assistant. Answer health questions clearly.
Keep answers short and easy to understand.
"""

# OUR GOOD prompt is the SYSTEM_PROMPT defined earlier

def test_prompt(prompt, question, label):
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        system=prompt,
        messages=[{"role": "user", "content": question}]
    )
    reply = response.content[0].text
    display(Markdown(f"**{label}**\n\n{reply}\n\n---"))
    return reply


test_question = "Can I take 1000mg of paracetamol three times a day?"

print('Comparing prompt quality on a medication dosage question...')
print(f'Test question: "{test_question}"\n')

print('--- Bad Prompt Response ---')
r_bad = test_prompt(BAD_PROMPT, test_question, 'Bad Prompt (vague, no safety)')

print('--- Good Prompt Response ---')
r_good = test_prompt(SYSTEM_PROMPT, test_question, 'Good Prompt (with safety rules)')

## Summary

**What this task covered:**

**Prompt Engineering:**
- Defined the model's role, tone, and behavior through a structured system prompt
- Used explicit rules instead of vague instructions (e.g., "NEVER diagnose" not "be careful")
- Added format instructions to control response length and style
- Showed through comparison that bad prompts produce dangerous or low-quality responses

**Safety Layers:**
- Layer 1: Pre-API keyword filter catches emergencies instantly (no API call needed)
- Layer 2: Sensitive topic detector adds extra disclaimers automatically
- Layer 3: The system prompt itself instructs the model to be cautious and always defer to doctors
- Layer 4: Response cleaning removes any filler phrases that slip through

**LLM API Usage:**
- Used the Anthropic API (same pattern works for OpenAI by changing ~5 lines)
- Showed how conversation history works — must send full history each turn since LLMs have no memory
- Demonstrated multi-turn conversation with context carried forward

**Key takeaway:** In a real health application, the prompt design and safety filters are often more important than which model you use. A strong prompt on GPT-3.5 can be safer and more useful than a weak prompt on GPT-4.